# Baseline: BIGRec

Qwen2.5-1.5B con ajuste completo, entrenado para escribir el título del siguiente ítem y evaluado con decodificación restringida al catálogo (paper, Apéndice B). Es el pipeline de LatentR3 sin el token latente: la fila "w/o Reasoning" de la Tabla 2 da los mismos números que BIGRec en la Tabla 1.

Usa `D3Dataset` y `CFEnhancedLogitsProcessor` de [SAI990323/DecodingMatters](https://github.com/SAI990323/DecodingMatters) (commit `585c842`), sin modificar, y los hiperparámetros de los autores sin grilla. El código está en `bigrec_baseline.py`. Cada dataset entrena su propio modelo y se evalúa por separado (solo los de `DATASETS`). **Este notebook escribe archivos** (checkpoints, modelo y predicciones) en `checkpoints/bigrec/<categoría>/`, ignorado por git, y puede retomarse si se corta.

Las decisiones, el estado, los costos y los resultados están en [`README.md`](README.md). En local se ejecuta desde `notebooks/bigrec/`; en Colab monta Google Drive y guarda las salidas en `MyDrive/IIC3633/checkpoints/bigrec/`.

## Entorno y configuración

In [1]:
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not IN_COLAB:
    os.environ['HF_HUB_OFFLINE'] = '1'   # Qwen2.5-1.5B ya está en la caché local: no se descarga nada
    # Con una GPU integrada, PyTorch ve dos GPUs y el Trainer reparte el modelo entre ambas (DataParallel), lo que
    # agota la memoria. Se deja visible solo la dedicada (índice 0). Tiene que ir antes de importar torch; no se
    # combina con HIP_VISIBLE_DEVICES (README de la raíz)
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import gc
import json
import shutil
import pandas as pd
import torch
import transformers

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
DATASETS = ['toys_data']   # se entrenan y evalúan solo estos (nombres de `names`)
names = {'CDs_and_Vinyl': 'cd_data',
         'Musical_Instruments': 'music_data',
         'Toys_and_Games': 'toys_data',
         'Video_Games': 'games_data'}
categorias = {nombre: categoria for categoria, nombre in names.items()}
# Configuración de memoria (README.md, Cómo correrlo). La prueba rápida eligió micro-batch 4 con adamw_torch en CDs,
# pero en Toys (prompts de hasta 365 tokens, frente a 296 en CDs) se quedó sin memoria antes de terminar la
# primera época. Micro-batch 2 reduce a la mitad la memoria de los logits y adamw_torch_fused evita los temporales
# del optimizador. El batch efectivo sigue en 128 (acumulación 64) y el algoritmo es el mismo. Si se corre la
# prueba rápida, la sobrescribe.
CONFIG_MEMORIA = {'micro_batch': 2, 'checkpointing': True, 'optim': 'adamw_torch_fused'}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
import sys
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
codigo_d3 = os.path.abspath('DecodingMatters' if IN_COLAB else os.path.join('..', '..', 'DecodingMatters'))
latentr3 = os.path.abspath('repo_latentr3' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3'))
SALIDA = ('/content/drive/MyDrive/IIC3633/checkpoints/bigrec' if IN_COLAB
          else os.path.abspath(os.path.join('..', '..', 'checkpoints', 'bigrec')))
calc_py = os.path.join(latentr3, 'src', 'utils', 'calc.py')
# baseline_comun.py está en notebooks/ en local y en el directorio de trabajo en Colab
comun = os.path.abspath('.' if IN_COLAB else '..')
if comun not in sys.path:
    sys.path.insert(0, comun)

In [3]:
for archivo in ['code/dataset.py', 'code/LogitProcesser.py']:
    if not os.path.isfile(os.path.join(codigo_d3, archivo)):
        raise FileNotFoundError(f'No se encontró {archivo} en {codigo_d3}. Descargar '
                                'https://github.com/SAI990323/DecodingMatters en esa carpeta y hacer checkout del commit 585c842.')
for archivo in [calc_py, os.path.join(latentr3, 'src', 'latent', 'reasoning_dataset.py')]:
    if not os.path.isfile(archivo):
        raise FileNotFoundError(f'No se encontró {archivo}. En Colab hay que subir repo_latentr3/src/.')
for modulo in ['bigrec_baseline.py', os.path.join(comun, 'baseline_comun.py')]:
    if not os.path.isfile(modulo):
        raise FileNotFoundError(f'No se encontró {os.path.abspath(modulo)}.')

import bigrec_baseline as bb
bb.cargar_d3(codigo_d3)   # verifica el sha256 antes de importar

print('torch', torch.__version__, '| transformers', transformers.__version__)
if transformers.__version__ != '4.51.3':
    print('AVISO: los autores usan transformers 4.51.3 y advierten que otra versión cambia los resultados (issue #2).')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
else:
    print('Sin GPU: solo se pueden correr las verificaciones sin GPU.')

torch 2.14.1+rocm7.14 | transformers 4.51.3
AMD Radeon RX 9070 XT | bf16: True


## Datos

Los mismos CSV que los demás baselines, sin re-split. Cada fila de `train` es una muestra: historial de ≤10 títulos → siguiente ítem. Se comparan los conteos con la Tabla 3 del paper (en CDs, `test` tiene 6157 filas y el paper dice 6158).

In [4]:
TABLA3 = {'toys_data': (53898, 6737, 6738, 6299), 'cd_data': (49251, 6156, 6158, 5841),
          'games_data': (75175, 9397, 9397, 5308), 'music_data': (66500, 8312, 8313, 5030)}
conteos = {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    fila = {split: len(pd.read_csv(bb.buscar_archivo(base, split, categoria))) for split in ['train', 'valid', 'test']}
    with open(bb.buscar_archivo(base, 'info', categoria), encoding='utf-8') as f:
        fila['ítems'] = sum(1 for _ in f)
    fila['Tabla 3 (train, valid, test, ítems)'] = TABLA3.get(nombre)
    conteos[nombre] = fila
pd.DataFrame(conteos).T

,train,valid,test,ítems,"Tabla 3 (train, valid, test, ítems)"
toys_data,53898,6737,6738,6299,"(53898, 6737, 6738, 6299)"


## Verificaciones sin GPU

**Prompt.** `bb.verificar_prompt` arma 200 filas de `train`, `valid` y `test` con `D3Dataset` y con `LatentRDataset` de LatentR3 (con `<|Thought|>` agregado a una copia del tokenizer, como hace LatentR3), y comprueba que los `input_ids`, `labels` y `attention_mask` sean iguales salvo la posición del token latente. Así se confirma que el prompt es el de LatentR3 menos el token. Las etiquetas valen −100 en todo el prompt: la loss solo cuenta el título objetivo, el salto de línea y el EOS. Se imprime un ejemplo de `train`, con las erratas originales ("musicss", "palyed").

**Trie.** La decodificación restringida usa un trie: un árbol con los tokens de todos los títulos del catálogo, donde cada prefijo apunta a los tokens que pueden seguir. `bb.verificar_trie` comprueba que:
- `### Response: \n` se tokeniza igual en todos los títulos y que todos los prompts de `test` terminan en esos 4 tokens, que son la raíz del trie;
- recorrer el trie da exactamente los títulos del catálogo, normalizados como `calc.py`, así que el modelo no puede generar nada fuera del catálogo;
- el título más largo cabe en `max_new_tokens` (128). También cuenta cuántos títulos cortaría el límite de 64 de D3 y cuántos contienen "Response:".

**Largos.** Tokens por fila (media, p99 y máximo) frente al `cutoff_len` de 512.

In [5]:
tokenizer = bb.cargar_tokenizer()
for nombre in DATASETS:
    ejemplo = bb.verificar_prompt(base, categorias[nombre], latentr3, tokenizer=tokenizer)
    print(f'{nombre}: prompt igual al de LatentR3 sin <|Thought|> en 200 filas de train, valid y test')
print('\nEjemplo de train (texto completo con el objetivo y EOS):\n')
print(ejemplo)

100%|██████████| 1/1 [00:00<00:00, 1474.79it/s]

toys_data: prompt igual al de LatentR3 sin <|Thought|> en 200 filas de train, valid y test

Ejemplo de train (texto completo con el objetivo y EOS):

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. 

### Instruction:
Given a list of toys and games the user recetenly enjoy, please write a new toys and games that the user may bought
### User Input: 
The user has palyed the following toys and gamess before: "Melissa & Doug Deluxe Easel Paper (3 pack bundle)", "ALEX Discover Paper Bag Puppets", "ALEX Discover Paper Bag Puppets", "Fisher-Price Loving Family SUV", "Barbie Dreamhouse"

### Response: 
"Barbie Dreamhouse"
<|endoftext|>


In [6]:
verificacion, largos = {}, {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    test = bb.construir_dataset(tokenizer, bb.buscar_archivo(base, 'test', categoria), categoria, test=True)
    verificacion[nombre] = bb.verificar_trie(bb.buscar_archivo(base, 'info', categoria), tokenizer,
                                             [test[i] for i in range(len(test))])
    train = bb.construir_dataset(tokenizer, bb.buscar_archivo(base, 'train', categoria), categoria)
    largos[(nombre, 'train: prompt + objetivo')] = bb.largos_tokens(train)
    largos[(nombre, 'test: prompt')] = bb.largos_tokens(test)
display(pd.DataFrame(verificacion).T)
pd.DataFrame(largos).T.round(1)

100%|██████████| 53898/53898 [00:16<00:00, 3204.03it/s]


,ítems,títulos únicos,caminos del trie,tokens del título más largo,títulos con más de 64 tokens (corte de D3),títulos con más de 128 tokens,títulos con 'Response:'
toys_data,6299,6271,6271,69,1,0,0


filas  media    p99  máximo
toys_data train: prompt + objetivo  53898.0  154.9  280.0   365.0
          test: prompt               6738.0  159.2  268.6   339.0

## Prueba rápida

Entrena en CDs con 1024 filas de `train` (8 pasos de 128 filas) y 256 de `valid`, en `checkpoints/bigrec/_prueba/`. La calidad del modelo no importa: sirve para medir memoria y tiempos.

**Memoria.** Los pesos, los gradientes y los dos estados de AdamW en bf16 ocupan gran parte de la GPU antes de las activaciones. Se prueban configuraciones de menor a mayor costo y se usa la primera que cabe. Ninguna cambia el batch efectivo de 128 ni el método:
1. **Gradient checkpointing**: no guarda los resultados intermedios del forward y los recalcula en el backward. El resultado es el mismo y es más lento.
2. **AdamW fused**: el mismo algoritmo en un solo kernel, sin los temporales de la versión `foreach`.
3. **Micro-batch 2 o 1**: menos filas por paso de la GPU y más acumulación (64 o 128). El Trainer normaliza la loss por el total de tokens de las 128 filas, así que el gradiente es el mismo salvo redondeo. La evaluación en `valid` sigue en lotes de 4, como los autores.

**Generación.** Evalúa 100 filas de `test` con lotes de 24 (como el script de LatentR3) y de 1 (como D3). Mide el tiempo por fila, comprueba que no haya títulos fuera del catálogo, que las métricas coincidan con `calc.py` y cuántas filas cambian su top-10 entre los dos tamaños de lote: con padding y bf16 puede haber diferencias de redondeo.

Con lo medido se estima la duración de cada dataset. Los outputs de estas celdas no quedaron guardados en el notebook (ver `README.md`, Costos medidos).

In [7]:
PRUEBA = os.path.join(SALIDA, '_prueba')
categoria = 'CDs_and_Vinyl'
CONFIGURACIONES = [   # de menor a mayor costo; se usa la primera que cabe (README.md, Adaptaciones)
    {'micro_batch': 4, 'checkpointing': True, 'optim': 'adamw_torch'},
    {'micro_batch': 4, 'checkpointing': True, 'optim': 'adamw_torch_fused'},
    {'micro_batch': 2, 'checkpointing': True, 'optim': 'adamw_torch_fused'},
    {'micro_batch': 1, 'checkpointing': True, 'optim': 'adamw_torch_fused'},
]
registro_prueba = None
for config in CONFIGURACIONES:
    shutil.rmtree(PRUEBA, ignore_errors=True)
    try:
        registro_prueba = bb.entrenar(categoria, base, PRUEBA, sample_train=1024, sample_valid=256, **config)
        CONFIG_MEMORIA = config
        break
    except torch.cuda.OutOfMemoryError:
        print('No cabe en la GPU:', config)
        gc.collect()
        torch.cuda.empty_cache()
assert registro_prueba is not None, 'Ninguna configuración cabe en la GPU'
print('Configuración elegida:', CONFIG_MEMORIA)
pd.Series({k: registro_prueba[k] for k in ['s_por_paso_mediana', 's_valid_por_fila', 'memoria_max_gb',
                                           'memoria_reservada_max_gb', 'micro_batch', 'acumulacion']})

100%|██████████| 256/256 [00:00<00:00, 3621.60it/s]
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Epoch,Training Loss,Validation Loss


[W1106 01:38:25.247890168 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 429916160 bytes (free: 216006656, total: 17095983104).
[W1106 01:38:27.881548285 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 564133888 bytes (free: 530579456, total: 17095983104).
[W1106 01:38:33.622340292 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 507510784 bytes (free: 304087040, total: 17095983104).


KeyboardInterrupt: 

In [ ]:
info = bb.buscar_archivo(base, 'info', categoria)
eval_prueba, predicciones = {}, {}
for carpeta, batch in [('bigrec_b24', bb.BATCH_GENERACION), ('bigrec_b1', 1)]:
    ruta = bb.evaluar_categoria(categoria, base, PRUEBA, 'bigrec', batch_size=batch, sample=100, carpeta=carpeta)
    with open(os.path.join(os.path.dirname(ruta), 'registro.json')) as f:
        registro = json.load(f)
    bb.verificar_con_calc(ruta, info, calc_py)   # falla si las métricas no coinciden con calc.py
    eval_prueba[carpeta] = {'s por fila': registro['segundos'] / registro['filas'],
                            'memoria máx. GB': registro.get('memoria_max_gb'), **bb.metricas(ruta, info)}
    with open(ruta) as f:
        predicciones[carpeta] = [fila['predict'] for fila in json.load(f)]
assert all(r['fuera del catálogo'] == 0 for r in eval_prueba.values())
distintas = sum(a != b for a, b in zip(predicciones['bigrec_b24'], predicciones['bigrec_b1']))
print(f'Filas con top-10 distinto entre lotes de 24 y de 1: {distintas} de {len(predicciones["bigrec_b1"])}')
pd.DataFrame(eval_prueba).T

In [ ]:
# Estimación por dataset con lo medido (la generación no se escala por el largo de los títulos)
filas_prueba = bb.construir_dataset(tokenizer, bb.buscar_archivo(base, 'train', categoria), categoria, sample=1024)
medicion = {'s_por_paso': registro_prueba['s_por_paso_mediana'], 's_valid_por_fila': registro_prueba['s_valid_por_fila'],
            'tokens_por_fila': bb.largos_tokens(filas_prueba)['media'],
            's_test_por_fila': eval_prueba['bigrec_b24']['s por fila']}
largos = globals().get('largos', {})   # lo llena la celda de verificaciones, si se corrió
estimaciones = {}
for nombre, cat in categorias.items():
    if (nombre, 'train: prompt + objetivo') not in largos:
        largos[(nombre, 'train: prompt + objetivo')] = bb.largos_tokens(
            bb.construir_dataset(tokenizer, bb.buscar_archivo(base, 'train', cat), cat))
    n = {split: len(pd.read_csv(bb.buscar_archivo(base, split, cat))) for split in ['train', 'valid', 'test']}
    estimaciones[nombre] = bb.estimar(n['train'], n['valid'], n['test'],
                                      largos[(nombre, 'train: prompt + objetivo')]['media'], medicion)
BORRAR_PRUEBA = True
if BORRAR_PRUEBA:
    shutil.rmtree(PRUEBA, ignore_errors=True)
pd.DataFrame(estimaciones).T.round(2)

## Entrenamiento

Un modelo por dataset, entrenado desde Qwen2.5-1.5B base. En cada época se recorren todas las filas de `train` en pasos de 128 filas (AdamW, lr 3e-4 con 20 pasos de warmup y bajada lineal), se calcula la loss en `valid` y se guarda un checkpoint. Para con paciencia 1 (si una época no mejora la loss de `valid`) o al llegar a 3 épocas, y se queda con la época de menor loss en `valid`. `test` no interviene.

Escribe en `checkpoints/bigrec/<categoría>/`:
- `trainer/`: checkpoints por época (~9 GB cada uno). Sirven para retomar si se corta: la celda continúa desde el último. Se borran al terminar.
- `modelo/`: la mejor época (~3,1 GB). Se conserva: la evaluación y D3 lo usan.
- `entrenamiento.json`: loss de `valid` por época, tiempos, memoria y versiones. Marca el dataset como terminado, así que volver a correr la celda lo salta.

Usa `CONFIG_MEMORIA`, que fija la prueba rápida.

In [5]:
assert CONFIG_MEMORIA is not None, 'Correr la prueba rápida o fijar CONFIG_MEMORIA en la celda de configuración'
registros = {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    registros[nombre] = bb.entrenar(categoria, base, os.path.join(SALIDA, categoria), **CONFIG_MEMORIA)
pd.DataFrame({nombre: {'eval_loss por época': [round(e['eval_loss'], 4) for e in r['evaluaciones']],
                       'mejor eval_loss': r['mejor_eval_loss'], 'pasos': r['pasos_totales'],
                       'h (última sesión)': r['segundos_esta_sesion'] / 3600, 'memoria máx. GB': r['memoria_max_gb'],
                       'micro-batch': r['micro_batch']}
              for nombre, r in registros.items()}).T

100%|██████████| 6737/6737 [00:02<00:00, 3180.09it/s]
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Epoch,Training Loss,Validation Loss
1,0.440600,0.730951
2,0.308300,0.625826


[W1106 02:44:59.469559712 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 662700032 bytes (free: 92274688, total: 17095983104).
[W1106 02:45:03.581774319 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 778043392 bytes (free: 222298112, total: 17095983104).
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


,eval_loss por época,mejor eval_loss,pasos,h (última sesión),memoria máx. GB,micro-batch
toys_data,"[0.731, 0.5748, 0.6258]",0.57483,1263,2.824073,12.996098,2


## Evaluación

Para cada fila de `test` se arma el prompt sin la respuesta y se generan 10 títulos con beam search de 10 beams: en cada paso se conservan los 10 comienzos de título más probables, y el trie solo permite tokens que continúan un título del catálogo. Los 10 títulos quedan ordenados por su log-probabilidad dividida por el largo (`length_penalty = 1`, como BIGRec en LatentR3). Así se ordena todo el catálogo, sin negativos muestreados.

Se guarda por partes de 256 filas en `predicciones/bigrec/`, así que se puede retomar. Al final se escribe `final_result.json` con el formato que lee `calc.py` de los autores.

In [6]:
for nombre in DATASETS:
    categoria = categorias[nombre]
    bb.evaluar_categoria(categoria, base, os.path.join(SALIDA, categoria), 'bigrec')

100%|██████████| 6738/6738 [00:02<00:00, 3062.09it/s]
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


Toys_and_Games (bigrec): parte 1/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 2/27, 0.25 s por fila
Toys_and_Games (bigrec): parte 3/27, 0.24 s por fila
Toys_and_Games (bigrec): parte 4/27, 0.26 s por fila


[W1106 04:43:23.735661716 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1247805440 bytes (free: 322961408, total: 17095983104).


Toys_and_Games (bigrec): parte 5/27, 0.23 s por fila
Toys_and_Games (bigrec): parte 6/27, 0.25 s por fila
Toys_and_Games (bigrec): parte 7/27, 0.25 s por fila
Toys_and_Games (bigrec): parte 8/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 9/27, 0.25 s por fila
Toys_and_Games (bigrec): parte 10/27, 0.25 s por fila


[W1106 04:49:46.508376189 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1264582656 bytes (free: 331350016, total: 17095983104).


Toys_and_Games (bigrec): parte 11/27, 0.27 s por fila
Toys_and_Games (bigrec): parte 12/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 13/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 14/27, 0.24 s por fila
Toys_and_Games (bigrec): parte 15/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 16/27, 0.25 s por fila
Toys_and_Games (bigrec): parte 17/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 18/27, 0.25 s por fila
Toys_and_Games (bigrec): parte 19/27, 0.27 s por fila
Toys_and_Games (bigrec): parte 20/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 21/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 22/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 23/27, 0.28 s por fila
Toys_and_Games (bigrec): parte 24/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 25/27, 0.26 s por fila


[W1106 05:06:38.098797174 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1308622848 bytes (free: 100663296, total: 17095983104).


Toys_and_Games (bigrec): parte 26/27, 0.26 s por fila
Toys_and_Games (bigrec): parte 27/27, 0.26 s por fila


## Métricas

HR@K y NDCG@K con el criterio de `calc.py`: hay acierto si el título generado, normalizado (`strip` de espacios, saltos de línea y comillas), es igual al título del objetivo. Si dos ítems comparten título, generar cualquiera cuenta como acierto. No hay criterio por id: un modelo generativo no distingue ítems con el mismo título.

Para cada dataset evaluado, `bb.verificar_con_calc` corre `gao` de `calc.py` sobre el mismo `final_result.json` y falla si da algo distinto. También se cuentan los títulos fuera del catálogo, que deben ser 0. La tabla usa todos los datasets que tengan predicciones, no solo los de `DATASETS`.

In [7]:
resultados = {}
for nombre, categoria in categorias.items():
    ruta = os.path.join(SALIDA, categoria, 'predicciones', 'bigrec', 'final_result.json')
    if not os.path.isfile(ruta):
        continue
    info = bb.buscar_archivo(base, 'info', categoria)
    bb.verificar_con_calc(ruta, info, calc_py)
    resultados[nombre] = bb.metricas(ruta, info)
print('Datasets evaluados:', list(resultados))
assert all(r['fuera del catálogo'] == 0 for r in resultados.values())
pd.DataFrame(resultados).T.round(4)

Datasets evaluados: ['toys_data']


,HR@1,HR@3,HR@5,HR@10,HR@20,NDCG@1,NDCG@3,NDCG@5,NDCG@10,NDCG@20,fuera del catálogo,repetidos,filas
toys_data,0.0352,0.0591,0.077,0.106,0.106,0.0352,0.049,0.0563,0.0657,0.0657,0.0,0.0,6738.0


## Comparación con el paper

- **Tabla 1, BIGRec**: los mismos números que "w/o Reasoning" de la Tabla 2 (Toys y CDs) y, en CDs, que "Constrained Decoding" de la Tabla 4. Una corrida determinista (Apéndice E).
- **Tabla 7, BIGRec en CDs**: media de 5 corridas con muestreo aleatorio en la generación. Sirve como referencia de la variación entre corridas.
- **Issue #2, CDs**: un usuario que corrió el SFT de LatentR3 con el script de los autores en 2 GPUs. **No es BIGRec** (tiene el token latente), pero muestra lo que se obtiene con este pipeline fuera del entorno de los autores.

In [8]:
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
PAPER = {'toys_data': [0.0701, 0.0931, 0.0508, 0.0582], 'cd_data': [0.0757, 0.0929, 0.0616, 0.0672],
         'games_data': [0.0461, 0.0709, 0.0334, 0.0414], 'music_data': [0.0938, 0.1158, 0.0807, 0.0879]}
REFERENCIAS = {'cd_data': [('Tabla 7: BIGRec, media de 5 corridas', [0.0725, 0.0902, 0.0595, 0.0653]),
                           ('Issue #2: LatentR3 only-SFT reproducido (no es BIGRec)',
                            [0.07698554, 0.09079097, 0.0639346, 0.06841407])]}
filas = []
for nombre, r in resultados.items():
    if nombre in PAPER:
        filas.append({'dataset': nombre, 'fuente': 'Tabla 1: BIGRec (= Tabla 2 w/o Reasoning)',
                      **dict(zip(METRICAS, PAPER[nombre]))})
    for fuente, valores in REFERENCIAS.get(nombre, []):
        filas.append({'dataset': nombre, 'fuente': fuente, **dict(zip(METRICAS, valores))})
    filas.append({'dataset': nombre, 'fuente': 'Nuestro BIGRec', **{m: r[m] for m in METRICAS}})
pd.DataFrame(filas).set_index(['dataset', 'fuente']).round(4)

HR@5   HR@10  NDCG@5  \
dataset   fuente                                                              
toys_data Tabla 1: BIGRec (= Tabla 2 w/o Reasoning)  0.0701  0.0931  0.0508   
          Nuestro BIGRec                             0.0770  0.1060  0.0563   

                                                     NDCG@10  
dataset   fuente                                              
toys_data Tabla 1: BIGRec (= Tabla 2 w/o Reasoning)   0.0582  
          Nuestro BIGRec                              0.0657